In [ ]:
import scipy.io as sio
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

dir = os.getcwd()+"/data/"
data = sio.loadmat(dir+"nn_data_rcn_kb.mat")
ws = data['ws']
phases = data['thetas']
wxs = data['wxs']
wxxs = data['wxxs']
wys = data['wys']
wyys = data['wyys']

nsamps,ny,nx = np.shape(ws)
perm = np.random.permutation(int(.9*nsamps))
training_samples = perm[:int(.8*nsamps)]
val_samples = perm[int(.8*nsamps):]
test_samples = np.arange(int(.9*nsamps), nsamps)

training_data = {'ws': ws[training_samples,:,:],
                 'wxs': wxs[training_samples,::4,::4],
                 'wxxs': wxxs[training_samples,::4,::4],
                  'wys': wys[training_samples,::4,::4],
                 'wyys': wyys[training_samples,::4,::4]}

val_data = {'ws': ws[val_samples,:,:],
                 'wxs': wxs[val_samples,::4,::4],
                 'wxxs': wxxs[val_samples,::4,::4],
                  'wys': wys[val_samples,::4,::4],
                 'wyys': wyys[val_samples,::4,::4]}

test_data = {'ws': ws[test_samples,:,:],
                 'wxs': wxs[test_samples,::4,::4],
                 'wxxs': wxxs[test_samples,::4,::4],
                  'wys': wys[test_samples,::4,::4],
                 'wyys': wyys[test_samples,::4,::4]}

w_train = training_data['ws']
w_test = test_data['ws']
w_val = val_data['ws']
w_train = w_train.astype('float32')
w_test = w_test.astype('float32')
w_val = w_val.astype('float32')
print(w_train.shape)
print(w_test.shape)
print(w_val.shape)

wx_train = training_data['wxs']
wx_train = wx_train.astype('float32')
wxx_train = training_data['wxxs']
wxx_train = wxx_train.astype('float32')
wy_train = training_data['wys']
wy_train = wy_train.astype('float32')
wyy_train = training_data['wyys']
wyy_train = wyy_train.astype('float32')



In [2]:
import tensorflow as tf
from tensorflow.keras import layers, losses, Model

class Autoencoder(Model):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = tf.keras.Sequential([
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dense(32, activation='relu'),
            layers.Dense(1024, activation='linear')
        ])
        self.decoder = tf.keras.Sequential([
            layers.Dense(64, activation='relu'),
            layers.Dense(784, activation='sigmoid'),
            layers.Dense(16384, activation='linear'),
            layers.Reshape((128, 128))# Adjust this based on your input dimensions
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def compute_loss(model, x, pre1, pre2, pre3, pre4):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x)
        reconstructed = model(x)
        # First derivative
        first_derivatives = tape.gradient(reconstructed, x)
    # Second derivatives
    second_derivatives = tape.gradient(first_derivatives, x)
    del tape  # Clean up the tape

    # Mean squared error loss
    mse_loss = tf.reduce_mean(tf.square(tf.subtract(reconstructed, x)))
    # Regularization from derivatives
    thetaxx = tf.multiply(second_derivatives,tf.square(pre1)) + tf.multiply(first_derivatives,pre2)
    thetayy = tf.multiply(second_derivatives,tf.square(pre3)) + tf.multiply(first_derivatives,pre4)
    divk = thetaxx + thetayy
    ksq = tf.square(pre1)+tf.square(pre3)
    kfth = tf.square(ksq)
    self_dual_rhs = tf.square(divk)-1+2*ksq -kfth
    sd_loss = tf.reduce_mean(tf.square(self_dual_rhs))
    total_loss = mse_loss + 0.1 * sd_loss
    return total_loss

# Training the model
@tf.function
def train_step(model, input, pre1, pre2, pre3, pre4, optimizer):
    with tf.GradientTape() as tape:
        loss = compute_loss(model, input, pre1, pre2, pre3, pre4)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss


2024-04-17 18:21:46.094950: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# # Example training loop
# def train(dataset, precomputed_data, epochs):
#     for epoch in range(epochs):
#         for (input, (pre1, pre2, pre3, pre4)) in zip(dataset, precomputed_data):
#             loss = train_step(autoencoder, input, pre1, pre2, pre3, pre4, optimizer)
#         print(f'Epoch {epoch + 1}, Loss: {loss.numpy()}')
#
# # def train(dataset, precomputed_data, epochs):
# #     for epoch in range(epochs):
# #         for (input, precomputed) in zip(dataset, precomputed_data):  # Adjust based on your dataset structure
# #             loss = train_step(autoencoder, input, precomputed, optimizer)
# #         print(f'Epoch {epoch + 1}, Loss: {loss.numpy()}')
#
# # Initializing the model and optimizer
# autoencoder = Autoencoder()
# optimizer = tf.keras.optimizers.Adam()
#
# batch_size = 32
# dataset = tf.data.Dataset.from_tensor_slices((w_train,
#                                               (wx_train, wxx_train, wy_train, wyy_train)))
# dataset = dataset.shuffle(buffer_size=1024).batch(batch_size)
#
# train(dataset, dataset, epochs=10)

In [4]:
# Batching the data using TensorFlow's Dataset API
batch_size = 32
main_dataset = tf.data.Dataset.from_tensor_slices(w_train).batch(batch_size)
precomputed_dataset = tf.data.Dataset.zip((
    tf.data.Dataset.from_tensor_slices(wx_train).batch(batch_size),
    tf.data.Dataset.from_tensor_slices(wxx_train).batch(batch_size),
    tf.data.Dataset.from_tensor_slices(wy_train).batch(batch_size),
    tf.data.Dataset.from_tensor_slices(wyy_train).batch(batch_size),
))

# Combine main data with precomputed data
combined_dataset = tf.data.Dataset.zip((main_dataset, precomputed_dataset))

# Define the model, optimizer, and train function if not already defined
autoencoder = Autoencoder()
optimizer = tf.keras.optimizers.Adam()

# Function to train the model
def train(dataset, epochs):
    for epoch in range(epochs):
        for batch in dataset:
            input_data, precomputed_data = batch
            loss = train_step(autoencoder, input_data, *precomputed_data, optimizer)
            print(f'Epoch {epoch + 1}, Loss: {loss.numpy()}')

# Start training
train(combined_dataset, epochs=10)

TypeError: 'NoneType' object is not callable